# 🛍️ RetailInsight — E-Commerce Sales Analysis

**Dataset:** Online Retail II (UCI Machine Learning Repository)  
**Source:** [Kaggle](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)  
**Goal:** Analyze sales trends, top products, revenue by country and customer behavior.

---

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='Blues_r')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully!')

Libraries loaded successfully!


## 2. Load Data

> Place `online_retail_II.csv` inside the `data/` folder.

In [2]:
df = pd.read_csv('data/online_retail_II.csv', encoding='latin-1', on_bad_lines='skip')
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 3. Data Overview

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [4]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [5]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

,Missing Count,Missing %
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00


## 4. Data Cleaning

In [6]:
# Drop rows with missing CustomerID and Description
df.dropna(subset=['Customer ID', 'Description'], inplace=True)

# Remove cancelled orders (Invoice starting with 'C')
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Remove negative or zero quantities and prices
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# Create Revenue column
df['Revenue'] = df['Quantity'] * df['Price']

# Extract date features
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Month']       = df['InvoiceDate'].dt.to_period('M')
df['DayOfWeek']   = df['InvoiceDate'].dt.day_name()
df['Hour']        = df['InvoiceDate'].dt.hour

print(f'Clean dataset shape: {df.shape}')
df.head()

Clean dataset shape: (805549, 12)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Month,DayOfWeek,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009-12,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009-12,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009-12,Tuesday,7


## 5. Revenue Trend Over Time

In [ ]:
monthly_revenue = df.groupby('Month')['Revenue'].sum().reset_index()
monthly_revenue['Month'] = monthly_revenue['Month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_revenue['Month'], monthly_revenue['Revenue'], marker='o', linewidth=2.5, color='#3b82f6')
ax.fill_between(monthly_revenue['Month'], monthly_revenue['Revenue'], alpha=0.15, color='#3b82f6')
ax.set_title('Monthly Revenue Trend', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('outputs/monthly_revenue.png', dpi=150)
plt.show()

## 6. Top 10 Countries by Revenue

In [ ]:
top_countries = (
    df[df['Country'] != 'United Kingdom']
    .groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_countries['Country'], top_countries['Revenue'], color='#3b82f6')
ax.bar_label(bars, fmt='£{:,.0f}', padding=5, fontsize=10)
ax.set_title('Top 10 Countries by Revenue (excl. UK)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Revenue (£)')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('outputs/top_countries.png', dpi=150)
plt.show()

## 7. Top 10 Best-Selling Products

In [ ]:
top_products = (
    df.groupby('Description')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_products['Description'], top_products['Quantity'], color='#60a5fa')
ax.bar_label(bars, fmt='{:,.0f}', padding=5, fontsize=10)
ax.set_title('Top 10 Best-Selling Products by Quantity', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Units Sold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('outputs/top_products.png', dpi=150)
plt.show()

## 8. Sales by Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Sunday']
day_revenue = (
    df.groupby('DayOfWeek')['Revenue']
    .sum()
    .reindex(day_order)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(day_revenue['DayOfWeek'], day_revenue['Revenue'], color='#3b82f6', edgecolor='white')
ax.set_title('Revenue by Day of Week', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Day')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('outputs/day_of_week.png', dpi=150)
plt.show()

## 9. Sales by Hour

In [ ]:
hourly = df.groupby('Hour')['Revenue'].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hourly['Hour'], hourly['Revenue'], marker='o', color='#3b82f6', linewidth=2.5)
ax.fill_between(hourly['Hour'], hourly['Revenue'], alpha=0.15, color='#3b82f6')
ax.set_title('Revenue by Hour of Day', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Hour')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig('outputs/hourly_sales.png', dpi=150)
plt.show()

## 10. Top 10 Customers by Revenue

In [ ]:
top_customers = (
    df.groupby('Customer ID')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_customers['Customer ID'] = top_customers['Customer ID'].astype(int).astype(str)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(top_customers['Customer ID'], top_customers['Revenue'], color='#3b82f6', edgecolor='white')
ax.bar_label(bars, fmt='£{:,.0f}', padding=5, fontsize=9)
ax.set_title('Top 10 Customers by Revenue', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Customer ID')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('outputs/top_customers.png', dpi=150)
plt.show()

## 11. Key Insights

---

In [13]:
total_revenue   = df['Revenue'].sum()
total_orders    = df['Invoice'].nunique()
total_customers = df['Customer ID'].nunique()
total_products  = df['Description'].nunique()
avg_order_value = total_revenue / total_orders

print('=' * 45)
print('         RETAIL INSIGHT — SUMMARY')
print('=' * 45)
print(f'  Total Revenue    : £{total_revenue:>12,.2f}')
print(f'  Total Orders     : {total_orders:>13,}')
print(f'  Unique Customers : {total_customers:>13,}')
print(f'  Unique Products  : {total_products:>13,}')
print(f'  Avg Order Value  : £{avg_order_value:>12,.2f}')
print('=' * 45)

         RETAIL INSIGHT — SUMMARY
  Total Revenue    : £17,743,429.18
  Total Orders     :        36,969
  Unique Customers :         5,878
  Unique Products  :         5,283
  Avg Order Value  : £      479.95


---
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn  
**Author:** Berke Arda Turk